LLM A Hands on appraoch project Gentut

In [1]:
!nvidia-smi

Sat Aug  1 04:11:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/GenTut', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/GenTut/hf_cache'  # cache model weights across sessions

In [ ]:
%cd /content/drive/MyDrive/GenTut
!git clone https://github.com/sandeshkg/gentut.git
%cd gentut

2. Install dependencies

In [4]:
%pip install -q langchain langgraph langchain-huggingface langchain-google-genai \
    transformers accelerate bitsandbytes pydantic streamlit google-generativeai python-dotenv

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

3. Hugging Face auth

In [ ]:
messages = [{"role": "user", "content": "Say hello in one sentence."}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
output = model.generate(inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))